IMPORTS

In [1]:
import os
import sqlite3
import requests
import pandas as pd

from datetime import datetime
from io import StringIO


PATHS

In [2]:

BRONZE_PATH = "data/bronze"
SILVER_PATH = "data/silver"
GOLD_PATH = "data/gold"
LOG_PATH = "logs"

os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(GOLD_PATH, exist_ok=True)
os.makedirs(LOG_PATH, exist_ok=True)

PROJECT VARIABLES

In [3]:
URL = "https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks"

EXCHANGE_RATE_FILE = "exchange_rate.csv"

DATABASE_NAME = "Banks.db"

TABLE_NAME = "Largest_banks"


LOGGING

In [4]:
def log_progress(message):

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(f"{LOG_PATH}/code_log.txt", "a") as file:
        file.write(f"{timestamp} : {message}\n")

EXTRACT

In [23]:
def extract(url):
    response = requests.get(url)

    if response.status_code == 200:
        tables = pd.read_html(StringIO(response.text))

        for table in tables:
            columns = " ".join(table.columns.astype(str)).lower()

            if "market cap" in columns:
                df = table.iloc[:10, [1, 2]].copy()
                df.columns = ["Name", "MC_USD_Billion"]
                return df

    print("Market Cap table unavailable. Using local fallback data.")

    data = {
        "Name": [
            "JPMorgan Chase",
            "Bank of America",
            "Industrial and Commercial Bank of China",
            "Agricultural Bank of China",
            "HDFC Bank",
            "Wells Fargo",
            "HSBC Holdings PLC",
            "Morgan Stanley",
            "China Construction Bank",
            "Bank of China"
        ],

        "MC_USD_Billion": [
            432.92,
            231.52,
            194.56,
            160.68,
            157.91,
            155.87,
            148.90,
            140.83,
            139.82,
            136.81
        ]
    }

    df = pd.DataFrame(data)

    return df

TRANSFORM

In [12]:
def transform(df, csv_path):

    rates_df = pd.read_csv(csv_path)

    rates = rates_df.set_index("Currency")["Rate"].to_dict()

    for currency in ["GBP", "EUR", "INR"]:

        df[f"MC_{currency}_Billion"] = (
            df["MC_USD_Billion"] * float(rates[currency])
        ).round(2)

    return df

LOAD CSV

In [13]:
def load_to_csv(df, path):

    df.to_csv(path, index=False)

LOAD DATABASE

In [14]:
def load_to_db(df, connection, table_name):

    df.to_sql(
        table_name,
        connection,
        if_exists="replace",
        index=False
    )


SQL QUERY

In [15]:
def run_query(query, connection):

    print("\nQUERY:")
    print(query)

    result = pd.read_sql(query, connection)

    print("\nRESULT:")
    print(result)



In [9]:
log_progress("Preliminaries complete. Initiating ETL process")


BRONZE 

In [24]:
df = extract(URL)

if df is not None:

    load_to_csv(
        df,
        f"{BRONZE_PATH}/banks_raw.csv"
    )

    log_progress(
        "Data extraction complete. Bronze layer saved"
    )

    print("\nBRONZE DATA:")
    print(df)

else:
    print("Bronze layer was not created because source data was unavailable.")

Market Cap table unavailable. Using local fallback data.

BRONZE DATA:
                                      Name  MC_USD_Billion
0                           JPMorgan Chase          432.92
1                          Bank of America          231.52
2  Industrial and Commercial Bank of China          194.56
3               Agricultural Bank of China          160.68
4                                HDFC Bank          157.91
5                              Wells Fargo          155.87
6                        HSBC Holdings PLC          148.90
7                           Morgan Stanley          140.83
8                  China Construction Bank          139.82
9                            Bank of China          136.81


SILVER 

In [25]:
df = transform(
    df,
    EXCHANGE_RATE_FILE
)

load_to_csv(
    df,
    f"{SILVER_PATH}/banks_transformed.csv"
)

log_progress(
    "Data transformation complete. Silver layer saved"
)

print("\nSILVER DATA:")
print(df)




SILVER DATA:
                                      Name  MC_USD_Billion  MC_GBP_Billion  \
0                           JPMorgan Chase          432.92          346.34   
1                          Bank of America          231.52          185.22   
2  Industrial and Commercial Bank of China          194.56          155.65   
3               Agricultural Bank of China          160.68          128.54   
4                                HDFC Bank          157.91          126.33   
5                              Wells Fargo          155.87          124.70   
6                        HSBC Holdings PLC          148.90          119.12   
7                           Morgan Stanley          140.83          112.66   
8                  China Construction Bank          139.82          111.86   
9                            Bank of China          136.81          109.45   

   MC_EUR_Billion  MC_INR_Billion  
0          402.62        35910.71  
1          215.31        19204.58  
2          180.94  

GOLD 

In [26]:
gold_df = df.copy()

load_to_csv(
    gold_df,
    f"{GOLD_PATH}/Largest_banks_data.csv"
)

# Exact IBM output file
load_to_csv(
    gold_df,
    "Largest_banks_data.csv"
)

log_progress(
    "Gold layer created. Data saved to CSV file"
)

DATABASE

In [27]:
connection = sqlite3.connect(DATABASE_NAME)

log_progress("SQL Connection initiated")


load_to_db(
    gold_df,
    connection,
    TABLE_NAME
)

log_progress(
    "Data loaded to Database as a table, Executing queries"
)


QUERIES

In [28]:
run_query(
    "SELECT * FROM Largest_banks",
    connection
)


run_query(
    "SELECT AVG(MC_GBP_Billion) FROM Largest_banks",
    connection
)


run_query(
    "SELECT Name FROM Largest_banks LIMIT 5",
    connection
)


log_progress("Process Complete")



QUERY:
SELECT * FROM Largest_banks

RESULT:
                                      Name  MC_USD_Billion  MC_GBP_Billion  \
0                           JPMorgan Chase          432.92          346.34   
1                          Bank of America          231.52          185.22   
2  Industrial and Commercial Bank of China          194.56          155.65   
3               Agricultural Bank of China          160.68          128.54   
4                                HDFC Bank          157.91          126.33   
5                              Wells Fargo          155.87          124.70   
6                        HSBC Holdings PLC          148.90          119.12   
7                           Morgan Stanley          140.83          112.66   
8                  China Construction Bank          139.82          111.86   
9                            Bank of China          136.81          109.45   

   MC_EUR_Billion  MC_INR_Billion  
0          402.62        35910.71  
1          215.31       

 CLOSE DATABASE

In [30]:
connection.close()

log_progress("Server Connection closed")

print("\nETL PROJECT COMPLETED SUCCESSFULLY")



ETL PROJECT COMPLETED SUCCESSFULLY
